In [ ]:
import nico 
from nico import Annotations as sann
from nico import Interactions as sint
from nico import Covariations as scov
import numpy as np
import os
import matplotlib.pyplot as plt 
from matplotlib.collections import PatchCollection

In [ ]:
import igraph
import scanpy as sc
import shutil

In [ ]:
ref_datapath='/cluster3/yflu/STS/TMA/celltypist/'
query_datapath="/cluster3/yflu/STS/TMA/tumor/"

In [ ]:
output_nico_dir='/cluster3/yflu/STS/TMA/tumor/'
output_annotation_dir=None #uses default location
#output_annotation_dir=output_nico_dir+'annotations/'
annotation_save_fname= 'tumor_nico.h5ad'
inputRadius=0

In [ ]:
ref_cluster_tag='Celltype_new' #scRNAseq cell type slot 
annotation_slot='nico_ct' #spatial cell type slot 

In [ ]:
shutil.copy("/cluster3/yflu/STS/TMA/TMA_tumor_pegasus_250508.h5ad","/cluster3/yflu/STS/TMA/tumor/sct_spatial.h5ad")

In [ ]:
anchors_and_neighbors_info=sann.find_anchor_cells_between_ref_and_query(
refpath=ref_datapath,
quepath=query_datapath,
output_nico_dir=output_nico_dir,
output_annotation_dir=output_annotation_dir)

In [ ]:
output_info=sann.nico_based_annotation(anchors_and_neighbors_info,
guiding_spatial_cluster_resolution_tag='louvain_labels',
across_spatial_clusters_dispersion_cutoff=0.15,
ref_cluster_tag=ref_cluster_tag,
resolved_tie_issue_with_weighted_nearest_neighbor='No') 

In [ ]:
sann.save_annotations_in_spatial_object(output_info,
anndata_object_name=annotation_save_fname)

In [ ]:
adata = sc.read(f'/cluster3/yflu/STS/TMA/tumor/tumor_nico.h5ad')

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color=["nico_ct"], show=False)
# Plot spatial data on the second axis

In [ ]:
adata.obs.nico_ct.value_counts()

In [ ]:
adata_1 = sc.read(f'/cluster3/yflu/STS/TMA/normal/normal_nico.h5ad')

In [ ]:
adata_1

In [ ]:
sc.pl.umap(adata_1, color=["celltype"], show=False,legend_loc = "on data")

In [ ]:
help(sc.pl.umap)

In [ ]:
sc.pl.umap(adata_1, color=["CD1C"], show=False,)

In [ ]:
data_T = sc.read("/cluster3/yflu/STS/TMA/TMA_T_pegasus_250503.h5ad")
data_B = sc.read("/cluster3/yflu/STS/TMA/TMA_B_pegasus_250503.h5ad")
data_M = sc.read("/cluster3/yflu/STS/TMA/TMA_M_pegasus_250503.h5ad")

In [ ]:
data_T.obs['nico_ct'] = adata_1.obs.loc[data_T.obs_names, 'nico_ct']
data_T.obs['nico_ct'] = data_T.obs['nico_ct'].cat.remove_unused_categories()
data_T.obs['nico_ct'].value_counts()
prop_df = (
    data_T.obs.groupby('louvain_labels_12')['nico_ct']
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
)

# Step 2: Identify dominant nico_ct per cluster
dominant_ct = prop_df.idxmax(axis=1)
dominant_ct_dict = dominant_ct.to_dict()

# Step 3: Add new column to adata.obs
data_T.obs['dominant_nico_ct'] = data_T.obs['louvain_labels_12'].map(dominant_ct_dict)

# Verify
print(data_T.obs[['louvain_labels_12', 'nico_ct', 'dominant_nico_ct']].head())

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(6, 15))  # Adjust figsize as needed

# Plot 1: louvain_labels_12
sc.pl.umap(
    data_T,
    color='louvain_labels_12',
    ax=axes[0],
    show=False,
    title='Clusters (louvain_labels_12)',
    legend_loc='right margin'  # Adjust legend position
)

# Plot 2: dominant_nico_ct
sc.pl.umap(
    data_T,
    color='dominant_nico_ct',
    ax=axes[1],
    show=False,
    title='Dominant Cell Type per Cluster',
    legend_loc='right margin'
)

# Plot 3: nico_ct
sc.pl.umap(
    data_T,
    color='nico_ct',
    ax=axes[2],
    show=False,
    title='All Cell Types (nico_ct)',
    legend_loc='right margin'
)

# Adjust spacing between subplots
plt.tight_layout()

# Save as PDF
plt.savefig('/cluster3/yflu/STS/TMA/normal/T_nico.pdf', bbox_inches='tight')
plt.close()  # Close the figure to free memory

In [ ]:
data_B.obs['nico_ct'] = adata_1.obs.loc[data_B.obs_names, 'nico_ct']
data_B.obs['nico_ct'] = data_B.obs['nico_ct'].cat.remove_unused_categories()
data_B.obs['nico_ct'].value_counts()
prop_df = (
    data_B.obs.groupby('louvain_labels_12')['nico_ct']
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
)

# Step 2: Identify dominant nico_ct per cluster
dominant_ct = prop_df.idxmax(axis=1)
dominant_ct_dict = dominant_ct.to_dict()

# Step 3: Add new column to adata.obs
data_B.obs['dominant_nico_ct'] = data_B.obs['louvain_labels_12'].map(dominant_ct_dict)

# Verify
print(data_B.obs[['louvain_labels_12', 'nico_ct', 'dominant_nico_ct']].head())

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(6, 15))  # Adjust figsize as needed

# Plot 1: louvain_labels_12
sc.pl.umap(
    data_B,
    color='louvain_labels_12',
    ax=axes[0],
    show=False,
    title='Clusters (louvain_labels_12)',
    legend_loc='right margin'  # Adjust legend position
)

# Plot 2: dominant_nico_ct
sc.pl.umap(
    data_B,
    color='dominant_nico_ct',
    ax=axes[1],
    show=False,
    title='Dominant Cell Type per Cluster',
    legend_loc='right margin'
)

# Plot 3: nico_ct
sc.pl.umap(
    data_B,
    color='nico_ct',
    ax=axes[2],
    show=False,
    title='All Cell Types (nico_ct)',
    legend_loc='right margin'
)

# Adjust spacing between subplots
plt.tight_layout()

# Save as PDF
plt.savefig('/cluster3/yflu/STS/TMA/normal/B_nico.pdf', bbox_inches='tight')
plt.close()  # Close the figure to free memory

In [ ]:
data_M.obs['nico_ct'] = adata_1.obs.loc[data_M.obs_names, 'nico_ct']
data_M.obs['nico_ct'] = data_M.obs['nico_ct'].cat.remove_unused_categories()
data_M.obs['nico_ct'].value_counts()
prop_df = (
    data_M.obs.groupby('louvain_labels_12')['nico_ct']
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
)

# Step 2: Identify dominant nico_ct per cluster
dominant_ct = prop_df.idxmax(axis=1)
dominant_ct_dict = dominant_ct.to_dict()

# Step 3: Add new column to adata.obs
data_M.obs['dominant_nico_ct'] = data_M.obs['louvain_labels'].map(dominant_ct_dict)

# Verify
print(data_M.obs[['louvain_labels_12', 'nico_ct', 'dominant_nico_ct']].head())
sc.pl.umap(data_M, color=['louvain_labels_12'], show=False)
sc.pl.umap(data_M, color=['nico_ct'], show=False)
sc.pl.umap(data_M, color=['dominant_nico_ct'], show=False)

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(6, 15))  # Adjust figsize as needed

# Plot 1: louvain_labels_12
sc.pl.umap(
    data_M,
    color='louvain_labels_12',
    ax=axes[0],
    show=False,
    title='Clusters (louvain_labels_12)',
    legend_loc='right margin'  # Adjust legend position
)

# Plot 2: dominant_nico_ct
sc.pl.umap(
    data_M,
    color='dominant_nico_ct',
    ax=axes[1],
    show=False,
    title='Dominant Cell Type per Cluster',
    legend_loc='right margin'
)

# Plot 3: nico_ct
sc.pl.umap(
    data_M,
    color='nico_ct',
    ax=axes[2],
    show=False,
    title='All Cell Types (nico_ct)',
    legend_loc='right margin'
)

# Adjust spacing between subplots
plt.tight_layout()

# Save as PDF
plt.savefig('/cluster3/yflu/STS/TMA/normal/M_nico.pdf', bbox_inches='tight')
plt.close()  # Close the figure to free memory